In [7]:
import trimesh
import numpy as np
import plotly.graph_objects as go
from plyfile import PlyData

scan_dir = "scan2therm/object_images/0cac754d-8d6f-2d13-8c81-d134a19b0045"

# Load mesh
mesh = trimesh.load(f"{scan_dir}/mesh.refined.v2.obj")
vertices = np.array(mesh.vertices)
faces = np.array(mesh.faces)

# Load instance labels
ply = PlyData.read(f"{scan_dir}/labels.instances.annotated.v2.ply")
obj_ids = ply['vertex']['objectId']

# Assign a random color per object
rng = np.random.RandomState(42)
unique_ids = np.unique(obj_ids)
color_map = {oid: rng.randint(50, 255, 3) for oid in unique_ids}
color_map[0] = np.array([200, 200, 200])  # background

min_v = min(len(obj_ids), len(vertices))
vertex_colors = np.array([color_map[oid] for oid in obj_ids[:min_v]])

# Build per-face color (use first vertex of each face)
face_colors = vertex_colors[faces[:, 0]]
face_color_strs = [f"rgb({r},{g},{b})" for r, g, b in face_colors]

print(f"Vertices: {len(vertices)}, Faces: {len(faces)}, Objects: {len(unique_ids)}")

Vertices: 44341, Faces: 65061, Objects: 20


In [8]:
fig = go.Figure(data=[
    go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        facecolor=face_color_strs,
        flatshading=True,
        lighting=dict(ambient=0.5, diffuse=0.5),
        lightposition=dict(x=0, y=0, z=2),
    )
])

fig.update_layout(
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    width=900, height=700,
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.show()